need to chnage this and to use only the consolidated. this is deprecated

In [0]:
%run ./helper

In [0]:
dbutils.widgets.dropdown("source_catalog", catalog_list[0], catalog_list)
dbutils.widgets.dropdown("source_schema", schema_list[0], schema_list)
dbutils.widgets.dropdown("model_result_table", model_result_table_list[0], model_result_table_list)
dbutils.widgets.dropdown("instruct_model", instruct_model_list[0], instruct_model_list)
catalog=dbutils.widgets.get("source_catalog")
schema=dbutils.widgets.get("source_schema")
model_result_table=dbutils.widgets.get("model_result_table")
instruct_model=dbutils.widgets.get("instruct_model")
df = spark.read.table(f"{catalog}.{schema}.{model_result_table}")
table_name = f"{catalog}.{schema}.{model_result_table}"
print(f"table_name:{table_name}, instruct_model:{instruct_model}" )


In [0]:
spark.table(table_name).schema

In [0]:
# 1. Configuration
model_id = instruct_model
column_name = f"ai_grade_{model_id.replace('databricks-', '').replace('-', '_').replace('_instruct', '')}"

# 2. Add the column if it doesn't exist
# This avoids errors during the update phase
cols = [f.name for f in spark.table(table_name).schema.fields]

if column_name not in cols:
    print(f"Column {column_name} missing. Adding it now...")
    spark.sql(f"ALTER TABLE {table_name} ADD COLUMN {column_name} STRING")
else:
    print(f"Column {column_name} already exists. Skipping creation.")

# 3. Define the AI Query Logic
# We use a f-string for the SQL but keep the inner JSON braces escaped
ai_query_logic = f"""
ai_query('{model_id}',
  concat(
    'Evaluate the cohesion of the following sentences and assign a relatedness score from 0 to 5. Output ONLY a valid JSON object. Do not include markdown blocks, prefixes, suffixes, or conversational text. The description must be an ultra-concise, direct justification in english (under 10 words) that goes straight to the point without starting with phrases like "The sentences are" or "They discuss". ',
    'Expected format: {{"score": integer, "description": "direct, focused reason in english"}}. ',
    'Sentences: ', 
    array_join(representative_Docs, ', '),
    ', represented by the words: ', 
    array_join(representation, ', ')
  )
)
"""

# 4. Execute the Incremental Update
# Only updates rows where the column is currently NULL
merge_sql = f"""
MERGE INTO {table_name} AS target
USING (
  SELECT 
    model, 
    timestamp, 
    topic,
    {ai_query_logic} AS new_grade
  FROM {table_name}
  WHERE {column_name} IS NULL
) AS source
ON target.model = source.model 
   AND target.timestamp = source.timestamp 
   AND target.topic = source.topic
WHEN MATCHED THEN
  UPDATE SET target.{column_name} = source.new_grade
"""

print(f"Running incremental AI evaluation for: {column_name}...")
spark.sql(merge_sql)
print("Update successful.")

In [0]:
spark.table(f"{catalog}.{schema}.topic_info_local").display()